<a href="https://colab.research.google.com/github/Albonire/clasificador-imagenes-openmp-cuda/blob/main/etapa2_cuda/cpu_baseline/base-model-cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline CPU — Clasificador de Somnolencia (MLP)
**Programacion Paralela y Computacion Distribuida · Universidad de Pamplona · 2026-I**

Este notebook entrena, **solo en CPU**, la misma red que luego se implementara con
kernels propios de CUDA en `etapa2_cuda`. Sirve como linea base de tiempo y exactitud
para calcular el *speedup* de la version GPU (mismo modelo, mismos datos, mismas epocas).

**Arquitectura (binaria, MLP de una sola capa oculta):**

```
Entrada(4096) -> Densa(oculta, ReLU) -> Densa(1, Sigmoide)
```

- **Perdida:** Binary Cross-Entropy (BCE)
- **Optimizador:** SGD (`peso = peso - tasa_aprendizaje * gradiente`)
- **Modo de entrenamiento:** full-batch (todo el train set por epoca), para que el
  tiempo de CPU sea comparable 1:1 con el kernel CUDA (que tambien procesara el
  dataset completo por epoca en lugar de mini-batches aleatorios).

**Contenido:**
1. Configuracion (entorno, semilla, forzar CPU)
2. Carga del dataset (`dataset/procesado/{train,val,test}.csv`)
3. Arquitectura del modelo (TensorFlow/Keras)
4. Busqueda de hiperparametros (epocas 20-50, lr 0.01-0.1, ocultas 64-128)
5. Entrenamiento final en CPU (tiempo medido)
6. Evaluacion en test
7. Exportacion de pesos (`W1, b1, W2, b2`, mismo formato que `app_streamlit`)
8. Reporte (`cpu_baseline_report.md`)


---
## 1. Configuración

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Albonire/clasificador-imagenes-openmp-cuda.git"
REPO_DIR = "/content/clasificador-imagenes-openmp-cuda"

if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if IN_COLAB:
    os.chdir(os.path.join(REPO_DIR, "etapa2_cuda", "cpu_baseline"))

print(f"En Colab: {IN_COLAB}")
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
!pip install tensorflow matplotlib scikit-learn

In [ ]:
import tensorflow as tf

import datetime
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Linea base de CPU: deshabilitar la GPU aunque el runtime la tenga disponible,
# para que el tiempo medido aqui sea estrictamente CPU-only.
tf.config.set_visible_devices([], "GPU")

print(f"TensorFlow version: {tf.__version__}")
print("Dispositivos visibles:")
for device in tf.config.get_visible_devices():
    print(f"  {device}")

In [ ]:
# --- Rutas del repositorio ---
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.exists(os.path.join(REPO_ROOT, "dataset")):
    REPO_ROOT = os.getcwd()

PATHS = {
    "procesado": os.path.join(REPO_ROOT, "dataset", "procesado"),
    "evidencias": os.path.join(REPO_ROOT, "reporte", "evidencias"),
    "baseline": os.path.join(REPO_ROOT, "etapa2_cuda", "cpu_baseline"),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {'OK' if os.path.isdir(path) else 'WARN'} {name}: {path}")

TRAIN_CSV = os.path.join(PATHS["procesado"], "train.csv")
VAL_CSV = os.path.join(PATHS["procesado"], "val.csv")
TEST_CSV = os.path.join(PATHS["procesado"], "test.csv")

print(f"\nTrain CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")
print(f"Test CSV:  {TEST_CSV}")

---
## 2. Carga del dataset

Particiones generadas en `notebooks/dataset_preparation.ipynb` (70/15/15,
estratificadas, sin overlap). Formato canonico: `label, pixel_0 .. pixel_4095`.

In [ ]:
TARGET = "label"


def load_split(path):
    df = pd.read_csv(path)
    feature_cols = [c for c in df.columns if c != TARGET]
    x = df[feature_cols].to_numpy(dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)
    return x, y


X_train, y_train = load_split(TRAIN_CSV)
X_val, y_val = load_split(VAL_CSV)
X_test, y_test = load_split(TEST_CSV)

N_FEATURES = X_train.shape[1]
assert N_FEATURES == 4096, f"Se esperaban 4096 features (64x64), se obtuvieron {N_FEATURES}"

for name, x, y in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    print(f"{name}: {x.shape}, positivos(clase1)={int(y.sum())} ({y.mean() * 100:.1f}%)")

print(f"\nFeatures: {N_FEATURES}")
print(f"Rango de pixeles: [{X_train.min():.4f}, {X_train.max():.4f}]")

---
## 3. Arquitectura del modelo (MLP)

Una sola capa oculta: suficiente para clasificacion binaria y evita programar la
convolucion hacia atras en CUDA (esa parte ya se hizo en OpenMP, etapa 1).

`Entrada(4096) -> Densa(oculta, ReLU) -> Densa(1, Sigmoide)`

Los kernels que reemplazaran cada pieza en CUDA:
- Densa(forward) → multiplicacion matriz-vector / matriz-matriz
- ReLU → suma de bias + ReLU
- Sigmoide → kernel de activacion de salida
- `binary_crossentropy` → kernel de perdida BCE
- `model.fit` (backprop) → kernels de gradiente (capa oculta y salida)
- `optimizers.SGD` → kernel de actualizacion de pesos (`peso -= lr * grad`)

In [ ]:
def build_model(hidden_units, learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(N_FEATURES,)),
        tf.keras.layers.Dense(hidden_units, activation="relu", name="hidden"),
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


build_model(64, 0.1).summary()

---
## 4. Busqueda de hiperparametros

Rango sugerido por la guia: epocas 20-50, tasa de aprendizaje 0.01-0.1,
neuronas ocultas 64-128. Se prueban los extremos de cada rango (grilla 2x2x2)
y se elige la combinacion con mejor `accuracy` en validacion.

In [ ]:
HIDDEN_GRID = [64, 128]
LR_GRID = [0.01, 0.1]
EPOCHS_GRID = [20, 50]

results = []

for hidden_units, learning_rate, epochs in itertools.product(HIDDEN_GRID, LR_GRID, EPOCHS_GRID):
    tf.random.set_seed(SEED)
    model = build_model(hidden_units, learning_rate)

    start = time.perf_counter()
    model.fit(
        X_train, y_train,
        batch_size=X_train.shape[0],  # full-batch, igual al kernel CUDA
        epochs=epochs,
        validation_data=(X_val, y_val),
        verbose=0,
    )
    elapsed = time.perf_counter() - start

    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    results.append({
        "hidden_units": hidden_units,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "train_time_sec": elapsed,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })
    print(
        f"hidden={hidden_units:3d} lr={learning_rate:<5} epochs={epochs:3d} "
        f"-> val_acc={val_accuracy:.4f} val_loss={val_loss:.4f} time={elapsed:.2f}s"
    )

results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
results_df

In [ ]:
best = results_df.iloc[0]
BEST_HIDDEN = int(best["hidden_units"])
BEST_LR = float(best["learning_rate"])
BEST_EPOCHS = int(best["epochs"])

print("Mejor combinacion (mayor accuracy en validacion):")
print(f"  hidden_units  = {BEST_HIDDEN}")
print(f"  learning_rate = {BEST_LR}")
print(f"  epochs        = {BEST_EPOCHS}")
print(f"  val_accuracy  = {best['val_accuracy']:.4f}")
print(f"  val_loss      = {best['val_loss']:.4f}")

---
## 5. Entrenamiento final en CPU

Se reentrena desde cero con la mejor combinacion y se mide el tiempo total con
`time.perf_counter()`. Este numero (`CPU_TRAIN_TIME_SEC`) es la linea base para el
speedup de la version CUDA.

In [ ]:
tf.random.set_seed(SEED)
final_model = build_model(BEST_HIDDEN, BEST_LR)

print(f"Entrenamiento final: hidden={BEST_HIDDEN}, lr={BEST_LR}, epochs={BEST_EPOCHS}")

start = time.perf_counter()
final_history = final_model.fit(
    X_train, y_train,
    batch_size=X_train.shape[0],
    epochs=BEST_EPOCHS,
    validation_data=(X_val, y_val),
    verbose=0,
)
CPU_TRAIN_TIME_SEC = time.perf_counter() - start

print(f"\nTiempo de entrenamiento en CPU: {CPU_TRAIN_TIME_SEC:.4f} s ({BEST_EPOCHS} epocas)")
print(f"Muestras de entrenamiento: {X_train.shape[0]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(final_history.history["loss"], label="train")
axes[0].plot(final_history.history["val_loss"], label="val")
axes[0].set_title("Perdida BCE")
axes[0].set_xlabel("Epoca")
axes[0].legend()

axes[1].plot(final_history.history["accuracy"], label="train")
axes[1].plot(final_history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoca")
axes[1].legend()

plt.tight_layout()
curves_path = os.path.join(PATHS["evidencias"], "cpu_baseline_training_curves.png")
fig.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {curves_path}")

---
## 6. Evaluacion en test

In [ ]:
y_prob = final_model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(np.int32)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Test accuracy:  {acc:.4f}")
print(f"Test precision: {prec:.4f}")
print(f"Test recall:    {rec:.4f}")
print(f"Test F1:        {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Abiertos (0)", "Cerrados (1)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Abiertos", "Cerrados"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Abiertos", "Cerrados"])
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusion - Test")
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, str(cm[i, j]), ha="center", va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

plt.tight_layout()
cm_path = os.path.join(PATHS["evidencias"], "cpu_baseline_confusion_matrix.png")
fig.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"  {cm_path}")

---
## 7. Exportacion de pesos

Mismo formato que espera `app_streamlit/app.py` (`W1, b1, W2, b2`, con
`W1: (oculta, entrada)` y `W2: (1, oculta)` para poder hacer `W @ x + b`).
Se guarda como checkpoint de esta linea base, sin sobrescribir
`modelo/weights.npz` (reservado para el modelo final entrenado en CUDA).

In [ ]:
w1_keras, b1_keras = final_model.get_layer("hidden").get_weights()
w2_keras, b2_keras = final_model.get_layer("output").get_weights()

W1 = w1_keras.T  # (oculta, entrada)
b1 = b1_keras  # (oculta,)
W2 = w2_keras.T  # (1, oculta)
b2 = b2_keras  # (1,)

weights_path = os.path.join(PATHS["baseline"], "weights_cpu_baseline.npz")
np.savez(weights_path, W1=W1, b1=b1, W2=W2, b2=b2)

print(f"Pesos guardados: {weights_path}")
print(f"  W1: {W1.shape}, b1: {b1.shape}, W2: {W2.shape}, b2: {b2.shape}")

---
## 8. Reporte

Resumen de la grilla de hiperparametros, la mejor combinacion, el tiempo de
entrenamiento en CPU y las metricas en test.

In [ ]:
report_path = os.path.join(PATHS["baseline"], "cpu_baseline_report.md")
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# CPU Baseline Report — Clasificador de Somnolencia\n\n")
    f.write(f"**Generado:** {now}\n\n")

    f.write("## Arquitectura\n\n")
    f.write(f"`Entrada({N_FEATURES}) -> Densa({BEST_HIDDEN}, ReLU) -> Densa(1, Sigmoide)`\n\n")
    f.write("- **Perdida:** Binary Cross-Entropy\n")
    f.write("- **Optimizador:** SGD (sin momentum)\n")
    f.write("- **Modo de entrenamiento:** full-batch (todo el train set por epoca)\n\n")

    f.write("## Busqueda de hiperparametros\n\n")
    f.write("| hidden_units | learning_rate | epochs | train_time_sec | val_loss | val_accuracy |\n")
    f.write("|---|---|---|---|---|---|\n")
    for _, row in results_df.iterrows():
        f.write(
            f"| {int(row['hidden_units'])} | {row['learning_rate']} | {int(row['epochs'])} | "
            f"{row['train_time_sec']:.4f} | {row['val_loss']:.4f} | {row['val_accuracy']:.4f} |\n"
        )
    f.write(f"\n**Mejor combinacion:** hidden={BEST_HIDDEN}, lr={BEST_LR}, epochs={BEST_EPOCHS}\n\n")

    f.write("## Entrenamiento final (CPU)\n\n")
    f.write(f"- **Tiempo de entrenamiento:** {CPU_TRAIN_TIME_SEC:.4f} s\n")
    f.write(f"- **Muestras de entrenamiento:** {X_train.shape[0]}\n")
    f.write(f"- **Epocas:** {BEST_EPOCHS}\n\n")

    f.write("## Metricas en test\n\n")
    f.write("| Metrica | Valor |\n|---------|-------|\n")
    f.write(f"| Accuracy | {acc:.4f} |\n")
    f.write(f"| Precision | {prec:.4f} |\n")
    f.write(f"| Recall | {rec:.4f} |\n")
    f.write(f"| F1 | {f1:.4f} |\n\n")

    f.write("## Uso\n\n")
    f.write(
        "`CPU_TRAIN_TIME_SEC` es la linea base para calcular el speedup de la "
        "implementacion CUDA en `etapa2_cuda` (mismo modelo, mismos datos, mismas epocas).\n"
    )

print(f"Reporte guardado: {report_path}")

---
## Siguiente paso

La implementacion en CUDA (`etapa2_cuda`) debe usar **la misma arquitectura y los
mismos hiperparametros** elegidos aqui (`hidden_units`, `learning_rate`, `epochs`)
y entrenar sobre los mismos `train.csv` / `val.csv` / `test.csv`, para que el
speedup GPU vs. CPU sea una comparacion valida. Pendiente medir ademas:

- `nvidia-smi` durante el entrenamiento en GPU (% utilizacion, memoria)
- Efecto del tamano de bloque (16×16, 32×32, ...) en los kernels CUDA